# EX 5 -- Mutual funds bought in lots (family 10)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/mip-modelling/blob/main/notebooks/ex05_funds.ipynb)

An integer (not binary) knapsack with only two lot types and a proportion
constraint rewritten in linear form. It also shows how a dual solution is checked:
an infeasible one is exhibited here first, as a
counterexample before the correct one is built.

The full chapter — model, data, results and sensitivity analysis — is [on the website](https://fabiofurini.github.io/mip-modelling/ex-05/).

## Setup

The cell below installs `gurobipy` and downloads the four shared modules of the
course: `stile.py` (palette), `mip.py` (relaxation, dual, bounds),
`euristiche.py` (next-fit, first-fit, best-fit) and `esteso.py` (the model of the
instance written out in full). The licence bundled with the pip package is limited
to **2000 variables and 2000 constraints**: the instances of the course are small
and all fit with plenty of room. For larger instances activate the free academic
licence at [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Environment: the solver and the shared modules of the course.
# Locally it uses the repository's python/stile.py; on Colab it installs and downloads what is missing.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

for modulo in ('stile', 'mip', 'euristiche', 'esteso', 'booleane'):                     # plotting style and course utilities
    locale = next((p for p in (Path(f"../python/{modulo}.py"), Path(f"python/{modulo}.py"))
                   if p.exists()), None)
    if locale is not None:
        sys.path.insert(0, str(locale.parent.resolve()))       # notebook opened in the repository
    else:                                 # on Colab always re-download: sessions last,
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/fabiofurini/mip-modelling/main/python/{modulo}.py", f"{modulo}.py")   # modules change

In [ ]:
import gurobipy as gp
import pandas as pd
from gurobipy import GRB

from mip import (ammissibile, due_rilassamenti, frazione, nuovo_modello, registra_bound,
                 risolvi, stampa_lp, valuta)
from stile import ARANCIO, BLU, GRIGIO, TEAL, intestazione, plt, salva_dati, salva_figura
from esteso import salva_modello

R = range

## 1. Model and instance

In [ ]:
intestazione("EX 5. Funds in lots: maximising the annual return within the budget")
c12 = [12, 15, 18, 25]         # cost of one lot (millions)
t12 = [1 / 6, 1 / 3, 1 / 9, 0.16]   # annual return, a fraction of the capital invested
nf = len(c12)
p12 = [c12[j] * t12[j] for j in R(nf)]  # return of one lot: 2, 5, 2 and 4 millions
B12 = 100                      # budget available
QUOTA = 0.5                    # fund 2 cannot exceed half of the total lots
# in the share constraint fund 2 has coefficient +1 and the others -1
QC = [1 if j == 1 else -1 for j in R(nf)]
salva_dati(pd.DataFrame({"fund": list(R(1, nf + 1)), "lot_cost": c12, "return": t12,
                         "lot_return": p12}), "ex05_dati")
print("  Return of one lot: "
      + ", ".join(f"fund {j + 1} = {c12[j]} * {frazione(t12[j])} = {frazione(p12[j])}"
                  for j in R(nf)) + " millions.")
print(f"  The constraint x2 <= {QUOTA} (x1 + x2 + x3 + x4) becomes, multiplying by 2 and "
      "moving to the left, -x1 + x2 - x3 - x4 <= 0.")


def modello(c, p, B):
    m = nuovo_modello("funds")
    x = m.addVars(nf, vtype=GRB.INTEGER, name="x")
    m.setObjective(gp.quicksum(p[j] * x[j] for j in R(nf)), GRB.MAXIMIZE)
    m.addConstr(gp.quicksum(c[j] * x[j] for j in R(nf)) <= B, name="budget")
    m.addConstr(gp.quicksum(QC[j] * x[j] for j in R(nf)) <= 0, name="share")
    return m, x


def duale(c, p, B):
    """min B alpha  s.t.  c_j alpha + QC_j beta >= p_j for every fund,  alpha, beta >= 0."""
    d = nuovo_modello("dual_funds")
    alpha = d.addVar(name="alpha")     # budget
    beta = d.addVar(name="beta")       # share
    d.setObjective(B * alpha, GRB.MINIMIZE)
    d.addConstrs((c[j] * alpha + QC[j] * beta >= p[j] for j in R(nf)), name="rc")
    return d


m12, x12 = modello(c12, p12, B12)
salva_modello(m12, "ex05_primale")
print("  The model of the instance:")
stampa_lp(m12)

## 2. Constructive heuristic (lower bound)

In [ ]:
# constructive heuristic on the return per million invested, respecting the share at every purchase
def euristica(c, p, B):
    x = [0] * nf
    ordine = sorted(R(nf), key=lambda j: (-p[j] / c[j], j))
    passi = ["return per million invested: "
             + ", ".join(f"fund {j + 1} = {frazione(p[j])}/{c[j]} = {frazione(p[j] / c[j])}"
                         for j in R(nf))
             + f"; we start from fund {ordine[0] + 1}"]
    for j in ordine:
        comprati = 0
        while True:
            prova = list(x)
            prova[j] += 1
            if (sum(c[k] * prova[k] for k in R(nf)) > B
                    or sum(QC[k] * prova[k] for k in R(nf)) > 0):
                break
            x, comprati = prova, comprati + 1
        residuo = B - sum(c[k] * x[k] for k in R(nf))
        motivo = ("the remaining budget is not enough for another lot"
                  if residuo < c[j] else "another lot would violate the share")
        passi.append(f"fund {j + 1}: {comprati} lots are bought and we stop because "
                     f"{motivo} ({residuo} millions left)")
    return x, passi


x_eur, passi = euristica(c12, p12, B12)
for k, riga in enumerate(passi, 1):
    print(f"  Step {k}. {riga}")
lb12 = sum(p12[j] * x_eur[j] for j in R(nf))
sol_eur = {f"x[{j}]": x_eur[j] for j in R(nf)}
assert ammissibile(m12, sol_eur), sol_eur
print(f"  Heuristic solution: {x_eur[0]} lots of fund 1 and {x_eur[1]} of fund 2   "
      f"lb = {frazione(lb12)}")

## 3. LP relaxation and dual (upper bound)

In [ ]:
d12 = duale(c12, p12, B12)
salva_modello(d12, "ex05_duale")
# counterexample: the choice alpha = 5/32, beta = 1/8 is not feasible
tentativo = {"alpha": 5 / 32, "beta": 1 / 8}
val_t, viol_t = valuta(d12, tentativo)
print(f"  A NOT feasible attempt: alpha = 5/32, beta = 1/8 gives "
      f"{c12[0]} * 5/32 - 1/8 = {frazione(c12[0] * 5 / 32 - 1 / 8)} < {frazione(p12[0])}: "
      f"the first dual constraint is violated by {frazione(viol_t)}.")
print("  A dual value may be read as a bound only after checking ALL the constraints.")
assert viol_t > 1e-9
# correct recipe: beta = 0 and alpha equal to the highest return per million
alpha_min = max(p12[j] / c12[j] for j in R(nf))
mano = {"alpha": alpha_min, "beta": 0.0}
ub12, viol = valuta(d12, mano)
assert viol <= 1e-9, viol
print(f"  Hand-built dual: beta = 0 and alpha = max_j p_j / c_j = {frazione(alpha_min)} "
      "(a million is worth what it returns in the best fund),")
print(f"  so every constraint c_j alpha >= p_j holds  ->  ub = {B12} * alpha = "
      f"{frazione(ub12)}")
zlp12, zlp12r, _ = due_rilassamenti(m12, d12)

## 4. Optimum of the MILP

In [ ]:
z12 = risolvi(m12)
print(f"  Optimal solution: {int(x12[0].X)} lots of fund 1 and {int(x12[1].X)} of fund 2, "
      f"spending {int(sum(c12[j] * x12[j].X for j in R(nf)))} out of {B12}, return "
      f"{frazione(z12)}")
riga = registra_bound("EX 5 funds", ub12, lb12, zlp12, zlp12r, z12, senso="max")
salva_dati(pd.DataFrame([riga]), "ex05_bound")
assert lb12 <= z12 <= zlp12 <= ub12 + 1e-9

## 5. The price of integrality

In [ ]:
intestazione("EX 5. The price of integrality and the role of the share")
print(f"  z(LP) = {frazione(zlp12)} against z(MILP) = {frazione(z12)}: the relaxation buys")
print(f"  {frazione(B12 / c12[0])} lots of fund 1, which cannot be bought in pieces.")
print(f"  The difference {frazione(zlp12 - z12)} is the cost of the indivisibility of the lots.")
print()
print("  The share bites: fund 2 returns more than twice the others per million invested,")
print("  and without the constraint one would buy that fund alone. With the share, every lot")
print("  of fund 2 has to be matched by a lot of some other fund:")
p_povero = [3.0 if j == 1 else p12[j] for j in R(nf)]   # fund 2 down to 20 per cent
prove = []
for nome, p_alt, quota in [("original data, with share", p12, True),
                           ("original data, without share", p12, False),
                           ("fund 2 less profitable, with share", p_povero, True),
                           ("fund 2 less profitable, without share", p_povero, False)]:
    m, x = modello(c12, p_alt, B12)
    if not quota:
        m.update()
        m.remove([c for c in m.getConstrs() if c.ConstrName == "share"][0])
        m.update()
    z = risolvi(m)
    print(f"  {nome:38s} z = {frazione(z):>4}   "
          f"x = ({', '.join(str(int(x[j].X)) for j in R(nf))})")
    prove.append({"variant": nome, "z": z}
                 | {f"x{j + 1}": int(x[j].X) for j in R(nf)})
salva_dati(pd.DataFrame(prove), "ex05_quota")
assert prove[2]["z"] < prove[3]["z"], "with a more profitable fund 2 the share must bite"

## 6. Figure: the solution

In [ ]:
# same look as the other figures of the chapter: one group of bars per fund,
# heuristic against optimum, and how much each solution commits
fig, ax = plt.subplots(figsize=(6.4, 3.0))
idx = list(R(nf))
ax.bar([j - 0.2 for j in idx], [x_eur[j] for j in idx], 0.4, color=ARANCIO,
       label=f"heuristic ({frazione(lb12)})")
ax.bar([j + 0.2 for j in idx], [x12[j].X for j in idx], 0.4, color=TEAL,
       label=f"optimum ({frazione(z12)})")
for j in idx:
    massimo = B12 // c12[j]
    ax.plot([j - 0.42, j + 0.42], [massimo, massimo], color=GRIGIO, lw=1.3, ls=":")
ax.plot([], [], color=GRIGIO, lw=1.3, ls=":", label="lots the budget alone would buy")
ax.set_xticks(idx)
ax.set_xticklabels([f"fund {j + 1}\n{c12[j]} mln a lot" for j in idx], fontsize=8)
ax.set_ylabel("lots bought")
ax.set_title(f"EX 5: the solution, {int(sum(c12[j] * x12[j].X for j in R(nf)))} millions "
             f"committed out of {B12}")
salva_figura(fig, "ex05_regione")
print("Done.")

---

Notebook generated from `python/ex05_funds.py` with `python3 python/make_notebooks.py`:
edits go into the script, not here.

Teaching material by [Fabio Furini](https://sites.google.com/view/fabiofurini/home-page) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/mip-modelling/blob/main/LICENSE),
code [MIT](https://github.com/fabiofurini/mip-modelling/blob/main/LICENSE-CODE).